# A first look at data: Big Tech in 2022
## Guided practice
ECON 3030 | CSU Bakersfield | Spring 2027

[Open this notebook in Google Colab](https://colab.research.google.com/github/allanwheeler/econ3030-spring2027/blob/main/notebooks/01-first-data-analysis.ipynb)

**Our question:** How did adjusted stock prices change for Apple, Microsoft, Meta, and NVIDIA during 2022?

You will load one CSV, inspect its rows, select columns, filter dates and companies, and calculate comparable price indexes with Polars. At the end, run a prepared Altair chart to see the paths you created. You do not need to write chart code yet.

These are real historical observations, not fictional prices. We are describing a selected period and four companies, not predicting prices or explaining what caused changes.

## Before you begin
Read the lecture introduction first. The earlier fictional study-hours example is a warm-up; this notebook applies data preparation to a real dataset.

1. Choose **File → Save a copy in Drive** to keep your work.
2. Read each explanation, make a prediction, and run the next code cell using its play button.
3. Work from top to bottom. A **code cell** contains Python instructions; a **text cell** contains explanations.

You need a Google account to save your copy, but no GitHub account or local software installation.

This is participation practice. A reasonable attempt in the designated exercise OR a completed reflection earns **Complete (100%)**, even if your code has errors. Correct results and successful execution are not required for participation credit.

## 1. Set up this session
A **library** is a collection of tools. Polars works with tables, Altair makes charts, and PyArrow helps exchange table data.

Run this setup cell once in each fresh Colab session. Wait for it to finish.

In [ ]:
%pip install -q "polars>=1.35,<2" "altair>=5.5,<6" "pyarrow>=26.0.1"

In [ ]:
import polars as pl
import altair as alt
from datetime import date

alt.renderers.enable("colab")

## 2. Load and inspect one table
The data come from [TidyTuesday, February 7, 2023](https://github.com/rfordatascience/tidytuesday/tree/main/data/2023/2023-02-07), which credits Yahoo Finance via Evan Gower's [Kaggle collection](https://www.kaggle.com/datasets/evangower/big-tech-stock-prices). A downloaded copy is stored at `data/teaching/big-tech-stock-prices.csv` in the course repository.

The cell uses the public upstream raw CSV so it works before the local course copy is published. No API or account is needed. Internet access is required.

A **CSV** stores a table as text. Each row here records a company on a recorded trading date, identified by `stock_symbol` and `date`. A stock symbol is a short company label: AAPL is Apple, MSFT is Microsoft, META is Meta, and NVDA is NVIDIA.

**Predict:** Will this table contain every calendar day, including weekends?

In [ ]:
data_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-02-07/big_tech_stock_prices.csv"
prices = pl.read_csv(data_url, try_parse_dates=True)
prices.head()

In [ ]:
prices.shape, prices.schema

`pl.read_csv` reads the table. `try_parse_dates=True` asks Polars to recognize dates. `prices` is a **variable**, a name we can reuse. `.head()` displays the first five rows. `.shape` reports rows and columns; `.schema` lists column names and their types.

The downloaded snapshot has 45,088 rows and eight columns covering 14 company symbols from January 2010 through January 2023, with different coverage by company. It has no blank cells, but that does not mean every trading date is present.

| Column | Meaning and unit |
| :--- | :--- |
| `stock_symbol` | Company trading symbol, a text label |
| `date` | Recorded trading date |
| `open`, `high`, `low`, `close` | Price measures in US dollars per share; see the source dictionary for adjustment details |
| `adj_close` | Closing price adjusted for splits and dividend distributions, in US dollars per share |
| `volume` | Number of shares traded |

We use `adj_close` consistently. Raw dollar prices are not directly comparable measures of company size. We will compare each company's change relative to its own starting value.

## 3. Keep the columns and observations we need
`select` chooses columns. `filter` keeps rows satisfying a condition. `pl.col` identifies a column by name.

Our window is January 3 through December 29, 2022. December 29 is the last common 2022 date available for these four companies in this snapshot; only Meta also has December 30. Using the same endpoints makes the comparison clearer.

`is_in` checks membership in a list, `is_between` checks a date range, and `&` means both conditions must hold. `sort` puts each company's dates in order.

**Predict:** Will selecting four companies reduce the number of rows or the number of columns?

In [ ]:
companies = ["AAPL", "MSFT", "META", "NVDA"]
start_date = date(2022, 1, 3)
end_date = date(2022, 12, 29)

selected_prices = (
    prices
    .select("stock_symbol", "date", "adj_close")
    .filter(
        pl.col("stock_symbol").is_in(companies)
        & pl.col("date").is_between(start_date, end_date)
    )
    .sort(["stock_symbol", "date"])
)
selected_prices.head()

## 4. Put each company on the same starting line
An **index** expresses a value relative to a starting point. Set each company's first adjusted price in our window to 100:

`price index = 100 × current adjusted price / first adjusted price`

An index of 80 means the adjusted price is 20% below its starting value; 120 means it is 20% above. This is not a dollar price or a forecast.

`with_columns` adds a column. `.first().over("stock_symbol")` supplies the first value separately for each company, using the date order we established. `.alias` gives the new column a name.

**Predict:** What should the first index value be for each company?

In [ ]:
indexed_prices = selected_prices.with_columns(
    (
        100 * pl.col("adj_close")
        / pl.col("adj_close").first().over("stock_symbol")
    ).alias("price_index")
)
indexed_prices.head()

## 5. Summarize the change
`group_by` collects rows for each company. `agg` calculates summaries within each group. `pl.len()` counts observations. Subtracting 100 from the final index gives the percentage change from the common starting date.

**Predict:** Does a final index below 100 imply a positive or negative change?

In [ ]:
summary = (
    indexed_prices
    .group_by("stock_symbol")
    .agg(
        pl.len().alias("observations"),
        pl.col("date").min().alias("first_date"),
        pl.col("date").max().alias("last_date"),
        pl.col("price_index").last().alias("final_index"),
    )
    .with_columns(
        (pl.col("final_index") - 100).round(1).alias("change_percent")
    )
    .sort("change_percent", descending=True)
)
summary

## 6. Visual Payoff: see the paths you prepared
Run the complete chart cell below. **The Altair code is supplied:** you are not expected to write it or understand every line yet. It uses `indexed_prices`, the Polars result you just created, without reloading or redoing the analysis.

Each line starts at 100. The dashed line marks that starting level. Point to a line for details; use the mouse wheel to zoom the date range and drag to move across time. The legend and hover labels identify companies.

**Look for:** Which company ends closest to 100? Does the final ranking tell you everything about what happened during the year?

In [ ]:
company_colors = alt.Scale(
    domain=["AAPL", "MSFT", "META", "NVDA"],
    range=["#0072B2", "#D55E00", "#009E73", "#CC79A7"],
)

lines = (
    alt.Chart(indexed_prices)
    .mark_line(strokeWidth=2.5)
    .encode(
        x=alt.X("date:T", title="Recorded trading date"),
        y=alt.Y("price_index:Q", title="Adjusted price index (start = 100)",
                scale=alt.Scale(zero=True)),
        color=alt.Color("stock_symbol:N", title="Company symbol", scale=company_colors),
        tooltip=[
            alt.Tooltip("stock_symbol:N", title="Company"),
            alt.Tooltip("date:T", title="Date"),
            alt.Tooltip("price_index:Q", title="Price index", format=".1f"),
        ],
    )
)
baseline = alt.Chart(pl.DataFrame({"start_index": [100]})).mark_rule(
    color="#666666", strokeDash=[5, 5]
).encode(y="start_index:Q")

chart = (
    (lines + baseline)
    .properties(
        title={
            "text": "Big Tech in 2022: four different paths from the same starting line",
            "subtitle": "January 3 to December 29 | Adjusted prices | Source: TidyTuesday / Yahoo Finance",
        },
        width=650,
        height=350,
        description="Adjusted stock price indexes for Apple, Microsoft, Meta, and NVIDIA. Each starts at 100; the summary table gives the final comparisons.",
    )
    .interactive(bind_y=False)
)
chart

### Interpret the chart carefully
Use the summary table as a text alternative to the final chart values. Describe a company's percentage change with the dates and measure. Check whether its final index matches the direction shown in the chart.

We selected four large technology companies and one period. This does not represent all stocks or establish what caused their changes. Adjusted prices reflect the source's split and dividend adjustments. The chart is not a guarantee of an investor's realized return and excludes taxes, fees, and other personal circumstances. Missing market dates and source revisions can affect comparisons. We have not independently audited Yahoo Finance's underlying records.

## 7. Participation exercise
**Small task:** try changing the company selection to include Netflix (`"NFLX"`) and calculate a filtered table for the same date window. Use `prices`, `start_date`, and `end_date` from above. You may adapt the filtering pattern in section 3.

Write your attempt in the designated code cell below, OR replace the reflection placeholder with what you tried or learned. Either meaningful response earns **Complete (100%)**, even with syntax errors or no successful execution. You do not have to do both. Merely running the supplied chart or keeping starter comments unchanged does not count.

Do not remove the designated cells or their identifying metadata.

In [ ]:
# Your participation attempt: try filtering prices to NFLX in the same date window.
# Add your own attempt below; this unchanged comment does not count.

### Optional participation reflection
Replace this placeholder with what you tried or learned, or a specific difficulty you encountered. This unchanged placeholder does not count.

## Submit your guided practice
Choose **File → Download → Download .ipynb** in Colab and upload the file to the LMS participation activity. Code errors and unexecuted cells are acceptable. The instructor compares the designated participation cells with the original starter, not your outputs.

The four graded labs are separate and use competency-based rubrics. Follow the LMS for those instructions; this notebook does not change Lab 1's Excel requirements.

## Help if you get stuck
- **Loading fails:** check internet access and rerun the load cell. The source is a public CSV, not an API. Ask the instructor if it remains unavailable.
- **A name is not defined:** run the preceding setup and preparation cells in order.
- **The chart is blank:** rerun the imports and prepared chart cell after the preparation steps. Confirm the filter did not remove all rows.
- **Colab restarts:** rerun setup and the preparation cells. Your saved notebook remains in Drive.

You can explain a difficulty in the participation reflection and still receive full participation credit.